# Step 4 — Data Analysis Prompts

**Project:** Prompt Engineering for Clothing Review Analysis  
**Dataset:** Women's Clothing E-Commerce Reviews (`data/reviews.csv`)  
**Goal:** Draft two analysis prompts (naive vs. improved), attach a fair sample of reviews, and compare the answers after pasting them into an LLM chat.

This notebook does **not** call any API. It prints copy-paste ready prompts. You paste each one into Claude or ChatGPT, then paste the model replies into the placeholder cells at the end.

## 0. Setup

Load the CSV the same way as in `01_data_exploration.ipynb`. The path works whether Jupyter was started from the project root (`data/reviews.csv`) or from inside `prompts/` (`../data/reviews.csv`).

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 120)

# Same resilient path as the EDA notebook
DATA_PATH = (
    Path("data/reviews.csv")
    if Path("data/reviews.csv").exists()
    else Path("..") / "data" / "reviews.csv"
)

df_raw = pd.read_csv(DATA_PATH, index_col=0)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

Loaded: C:\GamageRecruiters-DataScienceIntern\Month_02\prompt-engineering-task\data\reviews.csv
Raw shape: (23486, 10)


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comfortable,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap...",3,0,0,General,Dresses,Dresses


## 1. Clean the review text

The EDA notebook found two issues that would hurt prompt quality:

1. **845 rows** have no `Review Text`. Those rows cannot be analyzed as text, so we drop them.
2. **Row 646** (and similar rows) contain Windows line breaks (`\r\n`) inside the review, for example `\r\nproblem one: they bag out`. If we inject that raw text into a prompt, the line break can look messy. We replace `\r\n` with a space and strip leftover whitespace.

After cleaning, we keep only reviews that still have non-empty text.

In [2]:
# Work on a copy so the original load stays unchanged
df = df_raw.copy()

print("Rows before cleaning:", len(df))
print("Missing Review Text:", df["Review Text"].isna().sum())

# Drop reviews with no body text — they cannot go into a prompt
df = df.dropna(subset=["Review Text"])

def normalize_review_text(text):
    # Turn messy review text into a single clean string for prompt injection.
    # EDA finding (row 646): replace Windows line breaks with a space.
    text = str(text)
    text = text.replace("\r\n", " ")
    text = text.strip()
    return text

df["Review Text"] = df["Review Text"].map(normalize_review_text)

# If a review was only whitespace / line breaks, drop it too
df = df[df["Review Text"].str.len() > 0].copy()

print("Rows after cleaning:", len(df))

# Spot-check the EDA example (row 646)
if 646 in df.index:
    before = str(df_raw.loc[646, "Review Text"])
    after = df.loc[646, "Review Text"]
    print("Row 646 had \\r\\n before cleaning?", "\r\n" in before)
    print("Row 646 still has \\r\\n after cleaning?", "\r\n" in after)
    print("Row 646 after cleaning:", after[:180])
else:
    print("Row 646 was dropped (it had no usable review text).")


Rows before cleaning: 23486
Missing Review Text: 845
Rows after cleaning: 22641
Row 646 had \r\n before cleaning? True
Row 646 still has \r\n after cleaning? False
Row 646 after cleaning: I waited three months for these pants. when they finally arrived, i was mostly pleased. they seemed like the staple pant everyone is raving about. and they should have been. proble


## 2. Build a stratified analysis sample

A **random** sample of 20 reviews would be mostly 5-star, because that rating is the majority (EDA: about 56% of the data). Low-star complaints would almost disappear, and the model would look overly positive.

**Stratified sampling** means we take a similar number of reviews from *each* rating (1, 2, 3, 4, and 5). For `n=20` that is 4 reviews per star. A `seed` makes the sample repeatable: the same notebook run always picks the same rows.

`build_analysis_sample` returns a formatted **text block** (not a DataFrame) so it can be pasted straight into an LLM chat.

In [3]:
def build_analysis_sample(df, n=20, seed=42):
    """Return a formatted text block of n reviews, balanced across ratings 1-5.

    df    : cleaned reviews (must include Rating, Class Name, Recommended IND, Review Text)
    n     : total number of reviews to include (default 20)
    seed  : random seed so re-running the cell picks the same rows
    """
    ratings = [1, 2, 3, 4, 5]
    n_ratings = len(ratings)

    # Split n as evenly as possible across the 5 star values.
    # Example: n=20 -> 4 per rating. n=22 -> ratings 1 and 2 get 5, the rest get 4.
    base = n // n_ratings
    remainder = n % n_ratings

    pieces = []
    for i, rating in enumerate(ratings):
        # Give leftover slots to the lowest ratings first (they are the rarest)
        k = base + (1 if i < remainder else 0)
        group = df[df["Rating"] == rating]

        if group.empty:
            print(f"Warning: no reviews with Rating={rating}")
            continue

        # Do not request more rows than exist in this rating bucket
        k = min(k, len(group))
        pieces.append(group.sample(n=k, random_state=seed))

    sample = pd.concat(pieces, ignore_index=True)

    # Build a plain-text block the LLM can read
    lines = [
        f"Here are {len(sample)} clothing reviews for analysis.",
        "Each review includes Rating (1-5), Class Name, Recommended IND (1=yes, 0=no), and Review Text.",
        "",
    ]

    for i, row in sample.iterrows():
        rec_flag = row["Recommended IND"]
        rec_words = "recommended" if rec_flag == 1 else "not recommended"
        class_name = row["Class Name"] if pd.notna(row["Class Name"]) else "(Missing)"

        lines.append(f"--- Review {i + 1} ---")
        lines.append(f"Rating: {row['Rating']}")
        lines.append(f"Class Name: {class_name}")
        lines.append(f"Recommended IND: {rec_flag} ({rec_words})")
        lines.append(f"Review Text: {row['Review Text']}")
        lines.append("")

    return "\n".join(lines).strip(), sample


sample_text, sample_df = build_analysis_sample(df, n=20, seed=42)

print("Rows in sample:", len(sample_df))
print("Counts by Rating (should be ~equal):")
print(sample_df["Rating"].value_counts().sort_index())
print()
print("First 8 lines of the formatted block:")
print("\n".join(sample_text.splitlines()[:8]))


Rows in sample: 20
Counts by Rating (should be ~equal):
Rating
1    4
2    4
3    4
4    4
5    4
Name: count, dtype: int64

First 8 lines of the formatted block:
Here are 20 clothing reviews for analysis.
Each review includes Rating (1-5), Class Name, Recommended IND (1=yes, 0=no), and Review Text.

--- Review 1 ---
Rating: 1
Class Name: Dresses
Recommended IND: 0 (not recommended)
Review Text: If you have any curves, avoid! it is not flattering. the striped side panels look odd with the flow of the dress.


## 3. Draft two prompt versions

Prompt engineering is easier to see by contrast:

- **v1 (naive)** is vague. It does not say *who* the model is, *what* to look for, or *how* to format the answer. The model has to guess.
- **v2 (improved)** sets a role, a specific task, an output format, and a special instruction about rating vs. recommendation disagreements (a pattern we saw in EDA: a 3-star review can still be recommended).

Both prompts are stored as Python strings so we can attach the same sample text to each one.

In [4]:
# --- Version 1: vague on purpose (this is the "before" example) ---
prompt_v1_naive = "Analyze these reviews and tell me what's going on."

# --- Version 2: role + task + format + disagreement check ---
prompt_v2_improved = """
You are a product analytics assistant for a women's clothing retailer.

Task:
Read the customer reviews below. Identify:
1. The top 3 recurring complaints, grouped by theme (use themes such as fit, quality, or style).
2. The top 3 recurring praises, grouped by the same kinds of themes.

For each complaint and each praise:
- Name the theme.
- Write one short summary in your own words.
- Include one short example quote copied from a review.

Also note any reviews where Rating and Recommended IND disagree
(for example a low rating that is still recommended, or a high rating that is not recommended).
If none disagree, say so.

Output format:
Return two markdown tables, then a short disagreement note.

Table 1 — Top 3 recurring complaints
| Rank | Theme | Summary | Example quote |

Table 2 — Top 3 recurring praises
| Rank | Theme | Summary | Example quote |

Disagreement note:
A short bullet list of review numbers (or "None").
""".strip()

print("v1 length (characters):", len(prompt_v1_naive))
print("v2 length (characters):", len(prompt_v2_improved))
print()
print("v1 preview:", prompt_v1_naive)
print()
print("v2 first line:", prompt_v2_improved.splitlines()[0])


v1 length (characters): 50
v2 length (characters): 935

v1 preview: Analyze these reviews and tell me what's going on.

v2 first line: You are a product analytics assistant for a women's clothing retailer.


## 4. Full prompt v1 — copy this into Claude or ChatGPT

The cell below prints the **entire** naive prompt plus the 20-review sample. Select the cell output and copy it. Do not call an API from this notebook.

In [5]:
full_prompt_v1 = prompt_v1_naive + "\n\n" + sample_text

print("=" * 72)
print("COPY FROM HERE — PROMPT v1 (naive)")
print("=" * 72)
print(full_prompt_v1)
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)


COPY FROM HERE — PROMPT v1 (naive)
Analyze these reviews and tell me what's going on.

Here are 20 clothing reviews for analysis.
Each review includes Rating (1-5), Class Name, Recommended IND (1=yes, 0=no), and Review Text.

--- Review 1 ---
Rating: 1
Class Name: Dresses
Recommended IND: 0 (not recommended)
Review Text: If you have any curves, avoid! it is not flattering. the striped side panels look odd with the flow of the dress.

--- Review 2 ---
Rating: 1
Class Name: Jeans
Recommended IND: 0 (not recommended)
Review Text: I've purchased pilcro jeans in the past and they've held up great. unfortunately, the material used for this particular jean is a far, far cry from what it used to be. the thighs of these jeans start pilling (yes pilling!) after just a few wears. i've never had jeans do this -- whatever denim blend they are using is a complete disaster.  don't buy these -- you will be very disappointed.

--- Review 3 ---
Rating: 1
Class Name: Knits
Recommended IND: 0 (not recomme

## 5. Full prompt v2 — copy this into Claude or ChatGPT

Same sample as v1, but with the improved instructions. Using the **same reviews** makes the comparison fair: any difference in the answer comes from the prompt, not from different data.

In [6]:
full_prompt_v2 = prompt_v2_improved + "\n\n" + sample_text

print("=" * 72)
print("COPY FROM HERE — PROMPT v2 (improved)")
print("=" * 72)
print(full_prompt_v2)
print("=" * 72)
print("COPY TO HERE")
print("=" * 72)


COPY FROM HERE — PROMPT v2 (improved)
You are a product analytics assistant for a women's clothing retailer.

Task:
Read the customer reviews below. Identify:
1. The top 3 recurring complaints, grouped by theme (use themes such as fit, quality, or style).
2. The top 3 recurring praises, grouped by the same kinds of themes.

For each complaint and each praise:
- Name the theme.
- Write one short summary in your own words.
- Include one short example quote copied from a review.

Also note any reviews where Rating and Recommended IND disagree
(for example a low rating that is still recommended, or a high rating that is not recommended).
If none disagree, say so.

Output format:
Return two markdown tables, then a short disagreement note.

Table 1 — Top 3 recurring complaints
| Rank | Theme | Summary | Example quote |

Table 2 — Top 3 recurring praises
| Rank | Theme | Summary | Example quote |

Disagreement note:
A short bullet list of review numbers (or "None").

Here are 20 clothing revi

## Output — v1 (paste here)

Absolutely. Based on the reviews you provided, the main story is that **customer satisfaction is being driven by a few recurring product problems—especially fit, material quality, and durability—while positive reviews tend to emphasize comfort, appearance, and overall satisfaction.**

However, I can only see **Review 1–3 and the final snippet** in your message, not all 20 reviews. So I can give you a preliminary analysis now, but I can’t honestly analyze all 20 until you paste/upload the complete set.

### What is happening in the visible reviews

| Issue                        | Evidence                                                              | Likely impact                     |
| ---------------------------- | --------------------------------------------------------------------- | --------------------------------- |
| **Poor fit**                 | Dress is described as unflattering for curves; top initially ran huge | Strong reason for dissatisfaction |
| **Material quality**         | Jeans start pilling after only a few wears                            | Major quality complaint           |
| **Shrinkage after washing**  | Knitted top shrunk dramatically after one wash                        | Very strong negative experience   |
| **Comfort**                  | Final review says the shirt is “extremely comfortable”                | Positive driver                   |
| **Simple/attractive design** | Final review describes the shirt as simple and nice                   | Positive driver                   |

### Most important pattern

The negative reviews aren't simply saying **“I don't like the clothes.”** They point to **specific product-performance failures**:

* **Fit doesn't match expectations**
* **Fabric/material quality is poor**
* **Products don't maintain their quality after use or washing**
* Customers feel especially disappointed when the problem occurs **very quickly**, such as after a few wears or one wash.

For example, Review 2 is particularly concerning because the customer had previously purchased the same brand successfully. That means the complaint isn't only about the product—it suggests a **drop in perceived quality compared with previous products**.

Review 3 shows another important issue: the customer **liked the product initially**, but a washing-related problem completely changed the experience. So a product can have good design and initial appeal but still receive a **1-star rating because durability/maintenance performance fails**.

### Positive-review pattern

The visible positive review is much simpler:

> “Love the shirt... extremely comfortable, simple and nice.”

This suggests positive customers may value:

**Comfort + simplicity + attractive appearance → satisfaction/recommendation**

Whereas negative customers are often reacting to:

**Poor fit + poor material + poor durability/shrinkage → dissatisfaction/non-recommendation**

### What this could mean for your data-mining analysis

If these 20 reviews are part of a sentiment/review-mining exercise, I would investigate these **sentiment/topic categories**:

1. **Fit & sizing**
2. **Fabric/material quality**
3. **Durability**
4. **Shrinkage after washing**
5. **Comfort**
6. **Appearance/design**
7. **Value for money**
8. **Recommendation behavior**

A particularly interesting question is:

> **Which product issues are most strongly associated with a 1-star rating and “not recommended”?**

From the visible examples, **fit problems and quality/durability problems appear to be strong negative signals.**

If you send me the **full 20 reviews**, I can analyze them systematically and give you:

* overall sentiment
* most common complaints
* most common positive factors
* relationship between **Rating and Recommended IND**
* problems by **Class Name**
* keywords/topics associated with negative reviews
* which issues appear to cause **non-recommendation**
* a final **“What's going on?” business insight** suitable for a Data Science assignment.


## Output — v2 (paste here)

### Table 1 — Top 3 recurring complaints

| Rank | Theme | Summary                                         | Example quote |
| ---- | ----- | ----------------------------------------------- | ------------- |
| 1    | —     | No complaint identified in the provided review. | —             |
| 2    | —     | No complaint identified in the provided review. | —             |
| 3    | —     | No complaint identified in the provided review. | —             |

### Table 2 — Top 3 recurring praises

| Rank | Theme                | Summary                                                           | Example quote           |
| ---- | -------------------- | ----------------------------------------------------------------- | ----------------------- |
| 1    | Comfort              | The shirt is praised for being very comfortable to wear.          | “exteemely comfortable” |
| 2    | Style                | The simple and pleasant style is appreciated.                     | “simple and nice”       |
| 3    | Overall satisfaction | The customer expresses clear overall satisfaction with the shirt. | “Love the shirt.”       |

**Disagreement note:** No Rating or Recommended IND was provided for this review, so a disagreement cannot be identified.


## Comparison notes

- **What was vague/unclear in v1's output:**
  - The model provided a massive, unstructured response filled with unsolicited advice on how to do data mining, rather than just analyzing the data. 
  - It invented its own formatting and categories because the instructions were too broad. 

- **What specifically improved in v2:**
  - The output was strictly confined to the requested Markdown tables and headers.
  - Even though the data was inadvertently truncated during the copy-paste process (only showing the final review), the model strictly adhered to the requested structure, leaving the complaints table empty rather than hallucinating or rambling.

- **Why (what in the prompt caused the difference):**
  - Prompt v1 relied entirely on the LLM's default behavior by just asking it to "analyze."
  - Prompt v2 successfully controlled the AI's behavior by assigning a persona, defining the exact extraction targets (top 3 praises/complaints with quotes), and explicitly constraining the output to a specific Markdown format.
  - 